### Vocabulary Creation

Build vocabulary  sklearn. Print vocabulary size and analyze top frequent words.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

csv_path = Path('data/amazon_reviews_preprocessed.csv')
if not csv_path.exists():
    raise FileNotFoundError(f"Preprocessed dataset not found: {csv_path}")

df = pd.read_csv(csv_path)
if 'review_text_final' not in df.columns:
    raise KeyError("Expected column 'review_text_final' in the preprocessed dataset.")

texts = df['review_text_final'].fillna('').astype(str)
texts = texts[texts.str.strip() != '']

vectorizer = CountVectorizer(token_pattern=r'(?u)\b\w+\b')
bow_matrix = vectorizer.fit_transform(texts)

vocabulary = vectorizer.get_feature_names_out()
vocabulary_size = len(vocabulary)

word_frequencies = bow_matrix.sum(axis=0).A1
top_words_df = (
    pd.DataFrame({'word': vocabulary, 'frequency': word_frequencies})
    .sort_values(by='frequency', ascending=False)
    .reset_index(drop=True)
)

print(f"Number of reviews used: {len(texts)}")
print(f"Vocabulary size: {vocabulary_size}")
print("Top 20 most frequent words:")

top_words_df.head(20)


Number of reviews used: 688
Vocabulary size: 3251
Top 20 most frequent words:


,word,frequency
0,good,759
1,quality,683
2,sound,622
3,battery,501
4,earbuds,470
5,product,319
6,phone,318
7,bud,275
8,use,272
9,bass,257


In [2]:
# Export vocabulary artifacts
vocabulary_summary_df = (
    pd.DataFrame(
        {
            'word': vocabulary,
            'feature_index': [vectorizer.vocabulary_[word] for word in vocabulary],
            'frequency': word_frequencies,
        }
    )
    .sort_values(by='frequency', ascending=False)
    .reset_index(drop=True)
)

csv_export_path = Path('data/amazon_reviews_vocabulary.csv')
json_export_path = Path('data/amazon_reviews_vocabulary_mapping.json')

vocabulary_summary_df.to_csv(csv_export_path, index=False, encoding='utf-8')
pd.Series(vectorizer.vocabulary_).sort_index().to_json(json_export_path, force_ascii=False, indent=2)

print(f"Saved vocabulary summary to: {csv_export_path}")
print(f"Saved vocabulary mapping to: {json_export_path}")
print(f"Vocabulary size: {len(vocabulary_summary_df)}")

vocabulary_summary_df.head(10)


Saved vocabulary summary to: data/amazon_reviews_vocabulary.csv
Saved vocabulary mapping to: data/amazon_reviews_vocabulary_mapping.json
Vocabulary size: 3251


,word,feature_index,frequency
0,good,1314,759
1,quality,2296,683
2,sound,2707,622
3,battery,394,501
4,earbuds,942,470
5,product,2252,319
6,phone,2144,318
7,bud,498,275
8,use,3058,272
9,bass,390,257
